In [ ]:
import os
from pathlib import Path


def find_repo_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "downstream_tasks/expression_prediction").is_dir():
            return path
    raise RuntimeError("Run inside the GENA-LM clone or set GENA_HOME")


REPO_ROOT = (
    Path(os.environ["GENA_HOME"]).resolve()
    if "GENA_HOME" in os.environ
    else find_repo_root()
)
BENCHMARK_ROOT = Path(os.environ.get("BENCHMARK_ROOT", REPO_ROOT)).resolve()
TASK_ROOT = Path(os.environ.get("TASK_ROOT", REPO_ROOT)).resolve()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", REPO_ROOT / "data")).resolve()


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO = Path.cwd()
if not (REPO / "downstream_tasks").exists():
    REPO = REPO_ROOT

# BENCHMARK_ROOT is defined above
DATA = DATA_ROOT
GENA_ROOT = TASK_ROOT
ALPHAGENOME_ROOT = TASK_ROOT / "AlphaGenome"

sys.path.insert(0, str(REPO / "downstream_tasks/expression_prediction/benchmarks/gena_lm_benchmark/scripts"))
from score_ct_specificity import score_predictions


In [ ]:
model = "glioma"
split = "test"  # valid or test
cell_set = "json14"

pred_path = GENA_ROOT / "predictions_results" / model / f"gena_lm_{split}_{cell_set}_predictions.csv"
true_path = DATA / f"{split}_true_human.csv"
selected_targets_path = DATA / "selected_targets.csv"

pred = pd.read_csv(pred_path)
true = pd.read_csv(true_path)

if "gene_id" not in pred.columns:
    pred = pred.rename(columns={pred.columns[0]: "gene_id"})
if "gene_id" not in true.columns:
    true = true.rename(columns={true.columns[0]: "gene_id"})

print("pred:", pred.shape, pred_path)
print("true:", true.shape, true_path)
display(pred.head())
display(true.head())


In [ ]:
def correlation_table(true_df, pred_df):
    true = true_df.set_index("gene_id")
    pred = pred_df.set_index("gene_id")

    common_genes = true.index.intersection(pred.index)
    common_cells = true.columns.intersection(pred.columns)

    true = true.loc[common_genes, common_cells]
    pred = pred.loc[common_genes, common_cells]

    rows = []
    for cell in common_cells:
        true_vec = true[cell].astype(float).values
        pred_vec = pred[cell].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) < 2 or np.std(true_vec) == 0 or np.std(pred_vec) == 0:
            corr = np.nan
        else:
            corr = np.corrcoef(true_vec, pred_vec)[0, 1]
        rows.append({"cell_type": cell, "corr_genes": corr})

    result = pd.DataFrame(rows)

    gene_corrs = []
    skipped_genes = []
    for gene in common_genes:
        true_vec = true.loc[gene].astype(float).values
        pred_vec = pred.loc[gene].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) < 4 or np.std(true_vec) == 0 or np.std(pred_vec) == 0:
            skipped_genes.append(gene)
            continue
        corr = np.corrcoef(true_vec, pred_vec)[0, 1]
        if np.isfinite(corr):
            gene_corrs.append(corr)
        else:
            skipped_genes.append(gene)

    mean_corr_cells = float(np.mean(gene_corrs)) if gene_corrs else np.nan
    result["corr_cells"] = mean_corr_cells

    mean_row = pd.DataFrame([{
        "corr_genes": result["corr_genes"].mean(),
        "corr_cells": mean_corr_cells,
    }])
    result = pd.concat([result, mean_row], ignore_index=True)

    return result, true.reset_index(), pred.reset_index(), common_genes, common_cells, skipped_genes


In [ ]:
result, true_aligned, pred_aligned, common_genes, common_cells, skipped_genes = correlation_table(true, pred)
print("common genes:", len(common_genes))
print("common cells:", len(common_cells))
display(result)


In [ ]:
score_dict = score_predictions(true_aligned, pred_aligned, str(selected_targets_path), need_log=False)
deviation_r = score_dict.get("deviation_r", float("nan"))
print("deviation_r:", deviation_r)
